# QM9 `gap_eV` 予測 — 修正版（3方針の正式実装）

配布された `smiles` だけから特徴量を生成し、`gap_eV`（HOMO-LUMO ギャップ, eV）を予測する。
評価指標は **MAE**。全方針で **同一の 5-fold 交差検証**
（`KFold(n_splits=5, shuffle=True, random_state=8)`）を共有して公平に比較する。

## 当初決定した3方針（このノートブックで正しく実装する）

| 方針 | 特徴量 | モデル | リーク対策の要点 |
| --- | --- | --- | --- |
| **方針1** | 重要RDKit特徴量（標準2D記述子＋独自特徴量） | LightGBM / XGBoost（ブースティング） | 特徴量選択を**各学習fold内**で実行 |
| **方針2** | RDKit特徴量＋PCA | RBF-SVR | `SimpleImputer→StandardScaler→PCA→SVR` を **Pipeline** 化 |
| **方針3** | 大域RDKit＋独自特徴量＋Morgan FP | LightGBM / XGBoost | 高次元FPを列サンプリング＋正則化で過学習抑制 |

## 既存ノートブック（`qm9_gap_prediction.ipynb`）からの修正点
- 既存は「RDKit記述子＋LGBM」「記述子＋Morgan＋LGBM」のベースラインまでで、当初3方針とは別物だった → 3方針を正式実装。
- 既存LGBMは `n_estimators=5000` に対し `best_iteration_≒5000`（最大木数に張り付き、early stopping で収束していない）→ `learning_rate`・`n_estimators`・early stopping を再設定。
- ブレンドは方針3を置き換える形だった → ブレンドは §15 の**参考実験**へ移し、方針3は独立した正式方針にした。

## 記録済みベースライン（既存ノートブックの自己出力より・参考値）
- RDKit記述子 ＋ LightGBM：**CV MAE ≈ 0.2195**
- RDKit記述子 ＋ Morgan ＋ LightGBM：**CV MAE ≈ 0.2098**

これらは §10 のベースライン表に「参考（既存NB）」として明示する。再実行した値が異なる場合は
**実際の再実行結果を優先**する（値は捏造しない）。

## コンペ制約（順守事項）
- 外部データを追加しない（配布SMILESからの特徴量生成のみ）。
- 乱数シードは `8` に統一。
- テストの並び順は変更しない／予測値を手作業で変更しない。
- 提出は最大3モデル。各CSVは `smiles,gap_eV` の列順・4000行。

## 出力
`submission_strategy1.csv` / `submission_strategy2.csv` / `submission_strategy3.csv`

## 2. ライブラリ・設定

**何をする処理か**：必要ライブラリの読み込み、全乱数シードの固定（`8`）、実行モード（`QUICK`/`FULL`）と
探索設定・入出力パスの一元定義。

**なぜ必要か**：再現性の担保と、環境に応じた探索規模の切り替えのため。フル探索は数時間規模になり得るため、
まず `QUICK=True` で最後まで通し、値を確認してから `QUICK=False` でフル探索する運用を想定する。

**実行結果の読み方**：各ライブラリのバージョンが表示されれば環境準備は完了。バージョンは §18 で保存する。

In [1]:
import os, sys, json, time, random, hashlib, platform, warnings
from pathlib import Path
from typing import Callable, Dict, List, Sequence, Tuple, Optional

import numpy as np
import pandas as pd
import scipy.sparse as sp
import scipy.stats as ss
from scipy.optimize import minimize

import rdkit
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdMolDescriptors, rdFingerprintGenerator
from rdkit.Chem.rdchem import HybridizationType, BondType

import sklearn
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVR
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance

import lightgbm as lgb
import xgboost as xgb
import optuna
import shap
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
RDLogger.DisableLog("rdApp.*")
optuna.logging.set_verbosity(optuna.logging.WARNING)

LIB_VERSIONS = {
    "python": sys.version.split()[0], "platform": platform.platform(), "machine": platform.machine(),
    "numpy": np.__version__, "pandas": pd.__version__, "scipy": __import__("scipy").__version__,
    "scikit-learn": sklearn.__version__, "lightgbm": lgb.__version__, "xgboost": xgb.__version__,
    "optuna": optuna.__version__, "shap": shap.__version__, "rdkit": rdkit.__version__,
}
for k, v in LIB_VERSIONS.items():
    print(f"{k:14s}: {v}")

/Users/fu-riku/Library/CloudStorage/GoogleDrive-fukumoto.riku.fr7@g.ext.naist.jp/マイドライブ/MI_Lab_cloud/python-seminar/lesson_9/self-code/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


python        : 3.11.15
platform      : macOS-14.5-arm64-arm-64bit
machine       : arm64
numpy         : 2.4.6
pandas        : 3.0.3
scipy         : 1.17.1
scikit-learn  : 1.9.0
lightgbm      : 4.7.0
xgboost       : 3.2.0
optuna        : 4.9.0
shap          : 0.51.0
rdkit         : 2026.03.4


### 設定（シード・実行モード・探索規模）

- `QUICK=True`：Optuna試行数・SVR学習サンプル・設定スイープを縮小した**動作確認用**。
- `QUICK=False`：仕様どおりの**フル探索**（数時間規模になり得る）。

CPUのみ・GPU不要で動作する（LightGBM/XGBoost/SVR いずれもCPU実行、Apple Silicon 対応）。

In [2]:
SEED = 8
N_SPLITS = 5
QUICK = True   # ← フル探索するときは False にする

def set_all_seeds(seed: int) -> None:
    """Python / NumPy / ハッシュシードをまとめて固定する。"""
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_all_seeds(SEED)

DATA_DIR = Path(".")
TRAIN_PATH = DATA_DIR / "qm9_bandgap_train.csv"
TEST_PATH = DATA_DIR / "qm9_bandgap_test_without_answer.csv"
CACHE_DIR = Path("cache");   CACHE_DIR.mkdir(exist_ok=True)     # 特徴量キャッシュ
RESULTS_DIR = Path("results"); RESULTS_DIR.mkdir(exist_ok=True) # OOF/テスト予測・結果JSON

# 探索規模（QUICK/FULL 切り替え）
# 注: QM9のgapは滑らかな目的変数で、LightGBMの検証MAEは木数を増やしてもゆっくり改善し続ける
#     （lrを上げても best_iter は上限に張り付きやすい）。そこで木数に明確な上限を設けて計算量を抑え、
#     best_iter を常に出力して「上限張り付きか収束か」を可視化する（§11参照）。
CFG: Dict[str, object] = dict(
    boost_lr=0.05,
    boost_max_estimators=(500 if QUICK else 1500),          # baseline/重要度/スイープ用の木数上限
    boost_nest_range=((200, 700) if QUICK else (400, 2500)), # Optunaの n_estimators 探索範囲（計算量を有界化）
    early_stopping_rounds=(50 if QUICK else 100),
    # 方針1
    s1_feature_counts=([20, 80, "all"] if QUICK else [10, 20, 40, 80, 120, "all"]),
    s1_optuna_trials=(6 if QUICK else 30),
    s1_perm_repeats=(2 if QUICK else 5),
    s1_shap_sample=(400 if QUICK else 2000),
    # 方針2
    s2_pca_var=([0.95, 0.99] if QUICK else [0.90, 0.95, 0.97, 0.99]),
    s2_pca_ncomp=([40, 120] if QUICK else [20, 40, 80, 120]),
    s2_optuna_trials=(10 if QUICK else 40),
    s2_svr_max_train=(4000 if QUICK else None),   # QUICKはSVR学習をサブサンプル（fold毎）
    # 方針3
    s3_corr_thresholds=([0.98, None] if QUICK else [0.95, 0.98, 0.995, None]),  # None=高相関列を残す
    s3_morgan_variants=([("count", 2, 2048)] if QUICK
                        else [("count", 2, 2048), ("binary", 2, 2048), ("count", 3, 2048)]),
    s3_lowfreq_min_df=([1, 5] if QUICK else [1, 2, 5, 10]),  # 出現分子数の下限（1=未出現のみ削除）
    s3_optuna_trials=(6 if QUICK else 30),
    sweep_n_splits=(2 if QUICK else N_SPLITS),     # 設定スイープに使うfold数
)
print("QUICK =", QUICK)
print(json.dumps({k: v for k, v in CFG.items()}, ensure_ascii=False, indent=2, default=str))

QUICK = True
{
  "boost_lr": 0.05,
  "boost_max_estimators": 500,
  "boost_nest_range": [
    200,
    700
  ],
  "early_stopping_rounds": 50,
  "s1_feature_counts": [
    20,
    80,
    "all"
  ],
  "s1_optuna_trials": 6,
  "s1_perm_repeats": 2,
  "s1_shap_sample": 400,
  "s2_pca_var": [
    0.95,
    0.99
  ],
  "s2_pca_ncomp": [
    40,
    120
  ],
  "s2_optuna_trials": 10,
  "s2_svr_max_train": 4000,
  "s3_corr_thresholds": [
    0.98,
    null
  ],
  "s3_morgan_variants": [
    [
      "count",
      2,
      2048
    ]
  ],
  "s3_lowfreq_min_df": [
    1,
    5
  ],
  "s3_optuna_trials": 6,
  "sweep_n_splits": 2
}


### 共通ユーティリティ

**何をする処理か**：全方針で共有する関数（fold反復・ブースティングの学習/予測・結果保存・提出CSV検証など）を定義する。

**なぜ必要か**：全方針で**同一fold**・同一の早期終了条件・同一の検証手順を使い、リークと不整合を防ぐため。

**リーク上の注意**：early stopping は各学習foldの**内部分割**（外側検証foldは触らない）で行う。
`n_estimators` は上限として扱い、実際は early stopping が停止点を決める（既存NBの「最大木数張り付き」を回避）。

In [3]:
RESULTS: List[dict] = []   # §14 の比較表に集約する行

def iter_folds(fold_id: np.ndarray, n_splits: int):
    """fold_id 配列から (train_idx, valid_idx) を順に返す。全方針で共有。"""
    idx = np.arange(len(fold_id))
    for f in range(n_splits):
        yield idx[fold_id != f], idx[fold_id == f]

def make_lgb(params: Optional[dict] = None, seed: int = SEED) -> lgb.LGBMRegressor:
    base = dict(objective="regression_l1", metric="mae",
                n_estimators=CFG["boost_max_estimators"], learning_rate=CFG["boost_lr"],
                random_state=seed, n_jobs=-1, verbose=-1)
    if params:
        base.update(params)
    return lgb.LGBMRegressor(**base)

def make_xgb(params: Optional[dict] = None, seed: int = SEED) -> xgb.XGBRegressor:
    base = dict(objective="reg:absoluteerror", eval_metric="mae",
                n_estimators=CFG["boost_max_estimators"], learning_rate=CFG["boost_lr"],
                random_state=seed, n_jobs=-1, tree_method="hist",
                early_stopping_rounds=CFG["early_stopping_rounds"])
    if params:
        base.update(params)
    return xgb.XGBRegressor(**base)

def fit_boost_es(model, X_tr, y_tr, seed: int):
    """学習foldの内部分割(85/15)で early stopping して学習。外側検証foldは使わない（リーク回避）。"""
    xi, xv, yi, yv = train_test_split(X_tr, y_tr, test_size=0.15, random_state=seed)
    if isinstance(model, lgb.LGBMRegressor):
        model.fit(xi, yi, eval_set=[(xv, yv)], eval_metric="mae",
                  callbacks=[lgb.early_stopping(CFG["early_stopping_rounds"], verbose=False),
                             lgb.log_evaluation(0)])
    else:  # XGBRegressor（early_stopping_rounds はコンストラクタで指定済み）
        model.fit(xi, yi, eval_set=[(xv, yv)], verbose=False)
    return model

def boost_predict(model, X):
    """early stopping 済みモデルの推論（XGBは best_iteration までを使用）。"""
    if isinstance(model, xgb.XGBRegressor):
        bi = getattr(model, "best_iteration", None)
        if bi is not None:
            return model.predict(X, iteration_range=(0, bi + 1))
    return model.predict(X)

def cv_boost(get_fold_data, y, fold_id, kind: str, params: Optional[dict], seed: int = SEED):
    """共有foldでブースティングをCV。get_fold_data(f, tr, va) -> (X_tr_sub, X_va_sub, X_te)
    （行スライス済み。列はfoldごとに異なってよい）。戻り値: oof, test_pred(fold平均), fold_mae, best_iters。"""
    oof = np.zeros(len(y))
    test_pred = None
    fold_mae, best_iters = [], []
    for f, (tr, va) in enumerate(iter_folds(fold_id, N_SPLITS)):
        X_tr_sub, X_va_sub, X_te = get_fold_data(f, tr, va)
        model = make_lgb(params, seed) if kind == "lgb" else make_xgb(params, seed)
        fit_boost_es(model, X_tr_sub, y[tr], seed + f)
        oof[va] = boost_predict(model, X_va_sub)
        te = boost_predict(model, X_te)
        if test_pred is None:
            test_pred = np.zeros(len(te))
        test_pred += te / N_SPLITS
        fold_mae.append(mean_absolute_error(y[va], oof[va]))
        bi = getattr(model, "best_iteration_", None)
        if bi is None:
            bi = getattr(model, "best_iteration", None)
        best_iters.append(int(bi) if bi is not None else -1)
    return oof, test_pred, fold_mae, best_iters

def register_result(strategy, feature_set, model, n_features, fold_mae, train_time,
                    best_params, submission_path, oof=None, test_pred=None):
    """結果を RESULTS に追加し、OOF/テスト予測と要約JSONを results/ に保存する。"""
    fold_mae = list(map(float, fold_mae))
    try:
        n_features = int(n_features)
    except (TypeError, ValueError):
        pass  # "all" などの非数値はそのまま保持
    row = dict(strategy=strategy, feature_set=feature_set, model=model,
               n_features=n_features,
               cv_mae_mean=float(np.mean(fold_mae)), cv_mae_std=float(np.std(fold_mae)),
               **{f"fold{i+1}_mae": v for i, v in enumerate(fold_mae)},
               training_time=float(train_time), best_params=json.dumps(best_params, default=str),
               submission_path=submission_path)
    RESULTS.append(row)
    tag = strategy.replace(" ", "_").replace("/", "_")
    json.dump({**row, "lib_versions": LIB_VERSIONS}, open(RESULTS_DIR / f"{tag}.json", "w"),
              ensure_ascii=False, indent=2)
    if oof is not None:
        np.save(RESULTS_DIR / f"{tag}_oof.npy", np.asarray(oof))
    if test_pred is not None:
        np.save(RESULTS_DIR / f"{tag}_test.npy", np.asarray(test_pred))
    print(f"[登録] {strategy}: CV MAE = {row['cv_mae_mean']:.4f} ± {row['cv_mae_std']:.4f}")
    return row

def make_submission(pred: np.ndarray, filename: str, test_df: pd.DataFrame) -> pd.DataFrame:
    """test と同じ順序・行数で smiles,gap_eV のCSVを書き出し、妥当性を検証する。"""
    sub = pd.DataFrame({"smiles": test_df["smiles"].to_numpy(),
                        "gap_eV": np.asarray(pred, dtype=np.float64)})
    assert len(sub) == len(test_df), "行数がテストと不一致"
    assert list(sub.columns) == ["smiles", "gap_eV"], "列名/列順が不正"
    assert sub["smiles"].tolist() == test_df["smiles"].tolist(), "SMILES順序がテストと不一致"
    assert sub["gap_eV"].notna().all(), "欠損予測あり"
    assert np.isfinite(sub["gap_eV"].to_numpy()).all(), "無限値あり"
    assert not sub.duplicated().any(), "重複行あり"
    sub.to_csv(filename, index=False)
    print(f"保存: {filename}  ({len(sub)}行)")
    return sub

## 3. データ読み込み

**何をする処理か**：学習データ（`smiles`,`gap_eV`）とテストデータ（`smiles`）を読み込み、形状と目的変数分布を確認する。

**なぜ必要か**：以降の全処理の入力。列名・行数が想定どおりか（train 15000 / test 4000）を最初に確認する。

In [4]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
print("train:", train_df.shape, " test:", test_df.shape)
assert list(train_df.columns) == ["smiles", "gap_eV"], "train の列が想定と異なる"
assert list(test_df.columns) == ["smiles"], "test の列が想定と異なる"
y = train_df["gap_eV"].to_numpy(dtype=np.float64)
display(train_df.head(3))
print(train_df["gap_eV"].describe())

train: (15000, 2)  test: (4000, 1)


,smiles,gap_eV
0,C#CC1(CNC1=O)C#C,7.094012
1,CC1CC=CCOC=N1,6.854552
2,CC1=C2CCC3C(C1)C23,5.839566


count    15000.000000
mean         6.823883
std          1.288640
min          1.774183
25%          5.885826
50%          6.781081
75%          7.834162
max         10.718570
Name: gap_eV, dtype: float64


## 4. データ監査

**何をする処理か**：学習前に生データの健全性を確認する。SMILES欠損 / RDKit変換不能 / 重複 / canonical重複 /
同一canonicalに異なる `gap_eV` / 目的変数の欠損・分布 / train・testの元素構成差 / 分子サイズ分布差。

**なぜ必要か**：リークやデータ不整合を早期に発見するため。**無効SMILESは勝手に削除せず、行番号とSMILESを記録**する
（テスト側に変換不能があると4000行の予測が作れないため、その場合のみ明示的に停止）。

**実行結果の読み方**：`invalid`・`gap欠損`が0、canonical重複や `gap` 衝突が想定内であることを確認する。
特徴量レベルのNaN/inf/定数列/完全重複列/列不一致は §9 で点検する。

In [5]:
def smiles_to_mols(smiles: Sequence[str]) -> Tuple[List, List[Tuple[int, str]]]:
    """SMILES列を Mol へ変換し (mols, [(行番号, SMILES)] の無効リスト) を返す。行は削除しない。"""
    mols, invalid = [], []
    for i, smi in enumerate(smiles):
        m = Chem.MolFromSmiles(smi) if isinstance(smi, str) else None
        if m is None:
            invalid.append((i, smi))
        mols.append(m)
    return mols, invalid

train_mols, train_invalid = smiles_to_mols(train_df["smiles"].tolist())
test_mols, test_invalid = smiles_to_mols(test_df["smiles"].tolist())

audit = {}
audit["train_smiles_missing"] = int(train_df["smiles"].isna().sum())
audit["test_smiles_missing"] = int(test_df["smiles"].isna().sum())
audit["train_invalid_mol"] = len(train_invalid)
audit["test_invalid_mol"] = len(test_invalid)
audit["train_dup_smiles"] = int(train_df["smiles"].duplicated().sum())
audit["test_dup_smiles"] = int(test_df["smiles"].duplicated().sum())
audit["gap_missing"] = int(train_df["gap_eV"].isna().sum())

canon = [Chem.MolToSmiles(m) if m is not None else None for m in train_mols]
train_df = train_df.assign(_canon=canon)
audit["train_dup_canonical"] = int(train_df["_canon"].duplicated().sum())
gap_by_canon = train_df.dropna(subset=["_canon"]).groupby("_canon")["gap_eV"].nunique()
audit["canonical_gap_conflicts"] = int((gap_by_canon > 1).sum())

print(json.dumps(audit, ensure_ascii=False, indent=2))
if train_invalid or test_invalid:
    print("\n[無効SMILES 記録（削除はしない）]")
    for idx, smi in (train_invalid + test_invalid)[:20]:
        print("  row", idx, repr(smi))
assert not test_invalid, "テストに変換不能なSMILESがあります（4000行の予測が作れません）"

{
  "train_smiles_missing": 0,
  "test_smiles_missing": 0,
  "train_invalid_mol": 0,
  "test_invalid_mol": 0,
  "train_dup_smiles": 2,
  "test_dup_smiles": 0,
  "gap_missing": 0,
  "train_dup_canonical": 2,
  "canonical_gap_conflicts": 0
}


In [6]:
# 目的変数分布 / train-test の元素・サイズ分布差（監査の可視化は軽量な数値サマリで代替）
n_heavy_tr = np.array([m.GetNumHeavyAtoms() if m else 0 for m in train_mols])
n_heavy_te = np.array([m.GetNumHeavyAtoms() if m else 0 for m in test_mols])
print("重原子数  train: mean=%.2f min=%d max=%d | test: mean=%.2f min=%d max=%d"
      % (n_heavy_tr.mean(), n_heavy_tr.min(), n_heavy_tr.max(),
         n_heavy_te.mean(), n_heavy_te.min(), n_heavy_te.max()))

def element_composition(mols):
    from collections import Counter
    c = Counter()
    for m in mols:
        if m is None:
            continue
        for a in m.GetAtoms():
            if a.GetAtomicNum() > 1:
                c[a.GetSymbol()] += 1
    tot = sum(c.values()) or 1
    return {k: v / tot for k, v in sorted(c.items())}

comp_tr, comp_te = element_composition(train_mols), element_composition(test_mols)
comp = pd.DataFrame({"train_frac": comp_tr, "test_frac": comp_te}).fillna(0.0)
print("\n元素構成（重原子ベースの割合）:")
display(comp)
print("gap_eV: mean=%.3f std=%.3f min=%.3f max=%.3f"
      % (y.mean(), y.std(), y.min(), y.max()))

重原子数  train: mean=8.80 min=6 max=9 | test: mean=8.92 min=8 max=9

元素構成（重原子ベースの割合）:


,train_frac,test_frac
C,0.720207,0.736672
F,0.001704,0.000000
N,0.118256,0.099165
O,0.159833,0.164163


gap_eV: mean=6.824 std=1.289 min=1.774 max=10.719


## 5. 共通fold作成

**何をする処理か**：`KFold(n_splits=5, shuffle=True, random_state=8)` で各サンプルにfold番号を割り当て、`fold_id` として保存する。

**なぜ必要か**：**全方針・全ベースラインで同一の分割**を使い、CV MAE を公平に比較するため。以降は `iter_folds(fold_id, N_SPLITS)` を共有する。

In [7]:
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
fold_id = np.full(len(train_df), -1, dtype=int)
for f, (_, va) in enumerate(kf.split(train_df)):
    fold_id[va] = f
assert (fold_id >= 0).all()
np.save(RESULTS_DIR / "fold_id.npy", fold_id)
print("fold サイズ:", np.bincount(fold_id))

fold サイズ: [3000 3000 3000 3000 3000]


## 6. RDKit標準2D記述子生成

**何をする処理か**：`Descriptors.CalcMolDescriptors()` で標準2D記述子（約200個）を一括計算する。3次元コンフォマー・3D記述子は使わない。

**なぜ必要か**：分子量・環数・極性表面積など、化学的に解釈しやすい大域特徴量を得るため。記述子名一覧とRDKitバージョンは §9 のキャッシュに保存する。

In [8]:
def calc_descriptors(mols) -> pd.DataFrame:
    """RDKit標準2D記述子を一括計算して DataFrame で返す（None mol は空行→後段でNaN補完）。"""
    rows = [({} if m is None else Descriptors.CalcMolDescriptors(m))
            for m in tqdm(mols, desc="descriptors")]
    return pd.DataFrame(rows)

def file_signature(path: Path) -> dict:
    h = hashlib.sha1(Path(path).read_bytes()).hexdigest()
    return {"name": Path(path).name, "sha1": h}

def cached_frame(name: str, compute: Callable[[], pd.DataFrame], meta: dict) -> pd.DataFrame:
    """設定(meta)が一致すればParquetキャッシュを読み、変われば再計算して保存する。"""
    pq, mj = CACHE_DIR / f"{name}.parquet", CACHE_DIR / f"{name}.meta.json"
    if pq.exists() and mj.exists() and json.load(open(mj)) == meta:
        print(f"[cache hit] {name}")
        return pd.read_parquet(pq)
    df = compute()
    df.to_parquet(pq)
    json.dump(meta, open(mj, "w"), ensure_ascii=False, indent=2)
    print(f"[cache save] {name}  shape={df.shape}")
    return df

_desc_meta = dict(kind="rdkit_descriptors", rdkit=rdkit.__version__,
                  src=file_signature(TRAIN_PATH))
desc_train = cached_frame("desc_train", lambda: calc_descriptors(train_mols), _desc_meta)
_desc_meta_te = dict(_desc_meta, src=file_signature(TEST_PATH))
desc_test = cached_frame("desc_test", lambda: calc_descriptors(test_mols), _desc_meta_te)
DESC_NAMES = list(desc_train.columns)
json.dump({"rdkit": rdkit.__version__, "descriptor_names": DESC_NAMES},
          open(RESULTS_DIR / "descriptor_names.json", "w"), ensure_ascii=False, indent=2)
print("記述子数:", desc_train.shape[1])

descriptors: 100%|██████████| 15000/15000 [01:21<00:00, 183.76it/s]


[cache save] desc_train  shape=(15000, 217)


descriptors: 100%|██████████| 4000/4000 [00:20<00:00, 191.94it/s]


[cache save] desc_test  shape=(4000, 217)
記述子数: 217


## 7. 独自特徴量生成

**何をする処理か**：`Mol`/`Atom`/`Bond`/`RingInfo` から、元素・組成 / 原子状態 / 結合 / 環 / 比率の独自特徴量を生成する。
元素は固定せず、train・test に実在する重元素を走査して列を作る（QM9 では C/N/O/F）。比率はゼロ除算を防止する。

**なぜ必要か**：標準記述子が直接持たない、組成・混成・環種・形式電荷などの明示的な数え上げを補うため。

**リーク上の注意**：ここは目的変数を使わない純粋な構造特徴。列そのものにリークはない。

In [9]:
def get_heavy_element_set(*mol_lists) -> List[str]:
    """train・test に実在する重元素（原子番号>1）の記号集合を返す。"""
    syms = set()
    for mols in mol_lists:
        for m in mols:
            if m is None:
                continue
            for a in m.GetAtoms():
                if a.GetAtomicNum() > 1:
                    syms.add(a.GetSymbol())
    return sorted(syms)

def _safe_div(a: float, b: float) -> float:
    return float(a) / float(b) if b else 0.0

def calc_custom_features(m, elements: Sequence[str]) -> Dict[str, float]:
    """1分子の独自特徴量を dict で返す（None mol は空 dict）。"""
    if m is None:
        return {}
    atoms, bonds, ri = list(m.GetAtoms()), list(m.GetBonds()), m.GetRingInfo()
    n_heavy = m.GetNumHeavyAtoms()
    n_H = sum(a.GetTotalNumHs() for a in atoms) + sum(1 for a in atoms if a.GetAtomicNum() == 1)
    n_total = n_heavy + n_H
    elem_counts = {f"cnt_{s}": 0 for s in elements}
    n_hetero = n_aromatic_atom = n_ring_atom = n_sp = n_sp2 = n_sp3 = 0
    deg = {1: 0, 2: 0, 3: 0, 4: 0}
    fc_sum = fc_abs = n_pos = n_neg = 0
    for a in atoms:
        if a.GetAtomicNum() <= 1:
            continue
        s = a.GetSymbol()
        if f"cnt_{s}" in elem_counts:
            elem_counts[f"cnt_{s}"] += 1
        if s != "C":
            n_hetero += 1
        n_aromatic_atom += int(a.GetIsAromatic())
        n_ring_atom += int(a.IsInRing())
        hyb = a.GetHybridization()
        n_sp += int(hyb == HybridizationType.SP)
        n_sp2 += int(hyb == HybridizationType.SP2)
        n_sp3 += int(hyb == HybridizationType.SP3)
        d = a.GetDegree()
        if d in deg:
            deg[d] += 1
        fc = a.GetFormalCharge()
        fc_sum += fc; fc_abs += abs(fc)
        n_pos += int(fc > 0); n_neg += int(fc < 0)

    n_bonds = len(bonds)
    n_single = n_double = n_triple = n_arom_bond = n_conj = n_ring_bond = 0
    for b in bonds:
        bt = b.GetBondType()
        n_single += int(bt == BondType.SINGLE)
        n_double += int(bt == BondType.DOUBLE)
        n_triple += int(bt == BondType.TRIPLE)
        n_arom_bond += int(bt == BondType.AROMATIC or b.GetIsAromatic())
        n_conj += int(b.GetIsConjugated())
        n_ring_bond += int(b.IsInRing())

    ring_sizes = [len(r) for r in ri.AtomRings()]
    n_rings = len(ring_sizes)
    ring_size_counts = {f"ring_{k}": sum(1 for z in ring_sizes if z == k) for k in range(3, 9)}
    max_ring = max(ring_sizes) if ring_sizes else 0
    n_arom_ring = rdMolDescriptors.CalcNumAromaticRings(m)
    n_aliph_ring = rdMolDescriptors.CalcNumAliphaticRings(m)
    n_hetero_ring = (rdMolDescriptors.CalcNumAromaticHeterocycles(m)
                     + rdMolDescriptors.CalcNumAliphaticHeterocycles(m))
    n_carbo_ring = (rdMolDescriptors.CalcNumAromaticCarbocycles(m)
                    + rdMolDescriptors.CalcNumAliphaticCarbocycles(m))
    n_spiro = rdMolDescriptors.CalcNumSpiroAtoms(m)
    n_bridge = rdMolDescriptors.CalcNumBridgeheadAtoms(m)
    n_multiple = n_double + n_triple + n_arom_bond  # 多重結合 = 二重+三重+芳香族

    feat: Dict[str, float] = {k: float(v) for k, v in elem_counts.items()}
    feat.update(dict(
        cnt_H=float(n_H), n_heavy=float(n_heavy), n_total_atoms=float(n_total),
        n_hetero=float(n_hetero), fc_sum=float(fc_sum), fc_abs_sum=float(fc_abs),
        n_aromatic_atom=float(n_aromatic_atom), n_ring_atom=float(n_ring_atom),
        n_sp=float(n_sp), n_sp2=float(n_sp2), n_sp3=float(n_sp3),
        n_deg1=float(deg[1]), n_deg2=float(deg[2]), n_deg3=float(deg[3]), n_deg4=float(deg[4]),
        n_pos_charge=float(n_pos), n_neg_charge=float(n_neg),
        n_bonds=float(n_bonds), n_single=float(n_single), n_double=float(n_double),
        n_triple=float(n_triple), n_arom_bond=float(n_arom_bond), n_conj_bond=float(n_conj),
        n_ring_bond=float(n_ring_bond),
        n_rings=float(n_rings), max_ring_size=float(max_ring),
        n_aromatic_ring=float(n_arom_ring), n_aliphatic_ring=float(n_aliph_ring),
        n_hetero_ring=float(n_hetero_ring), n_carbo_ring=float(n_carbo_ring),
        n_spiro=float(n_spiro), n_bridgehead=float(n_bridge),
    ))
    feat.update({k: float(v) for k, v in ring_size_counts.items()})
    feat.update(dict(
        r_hetero=_safe_div(n_hetero, n_heavy), r_aromatic_atom=_safe_div(n_aromatic_atom, n_heavy),
        r_ring_atom=_safe_div(n_ring_atom, n_heavy), r_sp=_safe_div(n_sp, n_heavy),
        r_sp2=_safe_div(n_sp2, n_heavy), r_sp3=_safe_div(n_sp3, n_heavy),
        r_double=_safe_div(n_double, n_bonds), r_triple=_safe_div(n_triple, n_bonds),
        r_arom_bond=_safe_div(n_arom_bond, n_bonds), r_conj_bond=_safe_div(n_conj, n_bonds),
        r_multiple=_safe_div(n_multiple, n_bonds),
    ))
    return feat

def calc_custom_frame(mols, elements) -> pd.DataFrame:
    return pd.DataFrame([calc_custom_features(m, elements)
                         for m in tqdm(mols, desc="custom")])

ELEMENTS = get_heavy_element_set(train_mols, test_mols)
print("検出した重元素:", ELEMENTS)
_cust_meta = dict(kind="custom", rdkit=rdkit.__version__, elements=ELEMENTS,
                  src=file_signature(TRAIN_PATH))
cust_train = cached_frame("cust_train", lambda: calc_custom_frame(train_mols, ELEMENTS), _cust_meta)
cust_test = cached_frame("cust_test", lambda: calc_custom_frame(test_mols, ELEMENTS),
                         dict(_cust_meta, src=file_signature(TEST_PATH)))
print("独自特徴量数:", cust_train.shape[1])

検出した重元素: ['C', 'F', 'N', 'O']


custom: 100%|██████████| 15000/15000 [00:01<00:00, 13192.54it/s]


[cache save] cust_train  shape=(15000, 53)


custom: 100%|██████████| 4000/4000 [00:00<00:00, 13569.03it/s]


[cache save] cust_test  shape=(4000, 53)
独自特徴量数: 53


## 8. Morgan fingerprint 生成

**何をする処理か**：`rdFingerprintGenerator.GetMorganGenerator()` で Morgan FP を生成する。第一候補は
**count / radius=2 / fpSize=2048 / includeChirality=False**。方針3では count・binary、radius=2・3 も比較する。

**なぜ必要か**：部分構造の有無・個数を表す高次元特徴。方針3でブースティングに与える。

**メモリ**：FP は疎行列（CSR）で保持し、不要な dense 化を避ける。低頻度bitの削除条件（出現分子数の下限）は方針3で比較する。

In [10]:
def calc_morgan(mols, radius: int, nbits: int, count: bool) -> sp.csr_matrix:
    """Morgan FP を疎行列(CSR, float32)で返す。count=True で出現回数、False で 0/1。"""
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=nbits,
                                                    includeChirality=False)
    rows = []
    for m in tqdm(mols, desc=f"morgan(r{radius},{'cnt' if count else 'bin'})"):
        if m is None:
            rows.append(np.zeros(nbits, dtype=np.float32))
        elif count:
            rows.append(gen.GetCountFingerprintAsNumPy(m).astype(np.float32))
        else:
            rows.append(gen.GetFingerprintAsNumPy(m).astype(np.float32))
    return sp.csr_matrix(np.vstack(rows))

def cached_morgan(tag: str, mols, radius, nbits, count) -> sp.csr_matrix:
    """Morgan FP を .npz にキャッシュ（設定変更で自動再計算）。"""
    npz, mj = CACHE_DIR / f"morgan_{tag}.npz", CACHE_DIR / f"morgan_{tag}.meta.json"
    meta = dict(radius=radius, nbits=nbits, count=count, rdkit=rdkit.__version__,
                n=len(mols))
    if npz.exists() and mj.exists() and json.load(open(mj)) == meta:
        print(f"[cache hit] morgan_{tag}")
        return sp.load_npz(npz)
    M = calc_morgan(mols, radius, nbits, count)
    sp.save_npz(npz, M)
    json.dump(meta, open(mj, "w"))
    return M

# 第一候補（count, r2, 2048）を生成・キャッシュ
MFP_train = cached_morgan("train_c2_2048", train_mols, 2, 2048, True)
MFP_test = cached_morgan("test_c2_2048", test_mols, 2, 2048, True)
print("Morgan(count,r2,2048):", MFP_train.shape, "平均非ゼロ/分子:",
      round(MFP_train.nnz / MFP_train.shape[0], 2))

morgan(r2,cnt): 100%|██████████| 4000/4000 [00:00<00:00, 28488.83it/s]


Morgan(count,r2,2048): (15000, 2048) 平均非ゼロ/分子: 20.43


## 9. 特徴量キャッシュと整合性点検

**何をする処理か**：記述子＋独自特徴量を結合し、目的変数を使わない前処理（inf→NaN、train中央値補完、定数列削除、
完全重複列削除、train/test列整合）を行う。NaN/inf/定数列/完全重複列/列不一致を点検する。

**なぜ必要か**：欠損・定数・重複列は学習を不安定にする。これらの前処理は**教師なし**なので全trainで実施してもリークしない
（目的変数を使う特徴量選択は §11 で fold 内実施）。キャッシュには RDKitバージョン・設定・元CSVのハッシュを関連付け済み。

**実行結果の読み方**：`train/test 列一致 = True`、NaN/inf が 0、定数・重複で落ちた列数を確認する。

In [11]:
def clean_dense(train_feat: pd.DataFrame, test_feat: pd.DataFrame):
    """教師なし前処理：inf→NaN、train中央値補完、定数列・完全重複列の削除、train/test列整合。float32化。"""
    tr = train_feat.replace([np.inf, -np.inf], np.nan)
    te = test_feat.replace([np.inf, -np.inf], np.nan)
    med = tr.median(numeric_only=True)
    tr = tr.fillna(med); te = te.fillna(med)
    # 定数列削除
    keep = tr.columns[tr.nunique() > 1]
    dropped_const = [c for c in tr.columns if c not in set(keep)]
    tr = tr[keep]
    # 完全重複列削除（先に出た列を残す）
    trT = tr.T.drop_duplicates()
    dropped_dup = [c for c in tr.columns if c not in set(trT.index)]
    tr = trT.T
    te = te.reindex(columns=tr.columns, fill_value=0.0)
    return tr.astype(np.float32), te.astype(np.float32), dropped_const, dropped_dup

# 方針1・2 で使う「記述子＋独自特徴量」（完全重複列削除で標準記述子と重複する独自列も自動的に落ちる）
raw_train = pd.concat([desc_train, cust_train], axis=1)
raw_test = pd.concat([desc_test, cust_test], axis=1)
feat_dense, feat_dense_test, dropped_const, dropped_dup = clean_dense(raw_train, raw_test)
FEAT_NAMES = list(feat_dense.columns)

assert list(feat_dense.columns) == list(feat_dense_test.columns), "train/test 列不一致"
assert not feat_dense.isna().any().any() and not feat_dense_test.isna().any().any(), "NaN 残存"
assert np.isfinite(feat_dense.to_numpy()).all() and np.isfinite(feat_dense_test.to_numpy()).all(), "inf 残存"
print(f"結合前: {raw_train.shape[1]} 列 → 定数削除 {len(dropped_const)} / 完全重複削除 {len(dropped_dup)} → 採用 {len(FEAT_NAMES)} 列")
print("train/test 列一致:", list(feat_dense.columns) == list(feat_dense_test.columns))
X_dense = feat_dense.to_numpy(np.float32)
X_dense_test = feat_dense_test.to_numpy(np.float32)

結合前: 270 列 → 定数削除 31 / 完全重複削除 22 → 採用 217 列
train/test 列一致: True


## 10. ベースライン

**何をする処理か**：比較の下限として、(1) `gap_eV` 中央値予測、(2) 記述子＋Ridge、(3) 記述子＋RandomForest、
(4) 記述子＋LightGBM、(5) 記述子＋Morgan＋LightGBM を同一foldで評価する。Ridge は
`Imputer→Scaler→Ridge` の Pipeline（fold内fitでリーク回避）。

**なぜ必要か**：各方針が「単純な予測」や「素のブースティング」をどれだけ上回るかを測るため。

**実行結果の読み方**：中央値予測のMAEが上限の目安。既存NBの参考値（記述子+LGBM≈0.2195、+Morgan≈0.2098）と、
このNBの再実行値を並べて確認する（値が違えば再実行値を優先）。

In [12]:
def cv_sklearn(make_est: Callable[[], object], X_tr, X_te, y, fold_id, name):
    """共有foldで sklearn 推定器（毎foldで新規fit）をCV。oof, test_pred, fold_mae を返す。"""
    oof = np.zeros(len(y)); test_pred = np.zeros(X_te.shape[0]); fold_mae = []
    t0 = time.time()
    for tr, va in iter_folds(fold_id, N_SPLITS):
        est = make_est()
        est.fit(X_tr[tr], y[tr])
        oof[va] = est.predict(X_tr[va])
        test_pred += est.predict(X_te) / N_SPLITS
        fold_mae.append(mean_absolute_error(y[va], oof[va]))
    return oof, test_pred, fold_mae, time.time() - t0

# (1) 中央値予測
t0 = time.time(); oof_med = np.zeros(len(y)); mae_med = []
for tr, va in iter_folds(fold_id, N_SPLITS):
    med = np.median(y[tr]); oof_med[va] = med
    mae_med.append(mean_absolute_error(y[va], oof_med[va]))
register_result("baseline_median", "-", "median", 0, mae_med, time.time() - t0, {}, "-")

# (2) 記述子(+独自) + Ridge（Pipeline）
oof, tp, fm, tt = cv_sklearn(
    lambda: Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler()), ("ridge", Ridge(alpha=10.0, random_state=SEED))]),
    X_dense, X_dense_test, y, fold_id, "ridge")
register_result("baseline_ridge", "descriptors+custom", "Ridge", X_dense.shape[1], fm, tt, {"alpha": 10.0}, "-")

# (3) 記述子(+独自) + RandomForest
oof, tp, fm, tt = cv_sklearn(
    lambda: RandomForestRegressor(n_estimators=(200 if QUICK else 500), n_jobs=-1,
                                  random_state=SEED, criterion="absolute_error" if not QUICK else "squared_error"),
    X_dense, X_dense_test, y, fold_id, "rf")
register_result("baseline_rf", "descriptors+custom", "RandomForest", X_dense.shape[1], fm, tt, {}, "-")

# (4) 記述子(+独自) + LightGBM（early stopping を正しく設定）
oof, tp, fm, bi = cv_boost(
    lambda f, tr, va: (X_dense[tr], X_dense[va], X_dense_test), y, fold_id, "lgb", None)
print("  LightGBM best_iters:", bi)
register_result("baseline_lgbm_desc", "descriptors+custom", "LightGBM", X_dense.shape[1],
                fm, 0.0, {}, "-", oof=oof, test_pred=tp)

# (5) 記述子(+独自) + Morgan + LightGBM（疎行列で結合）
Xc_tr = sp.hstack([sp.csr_matrix(X_dense), MFP_train]).tocsr()
Xc_te = sp.hstack([sp.csr_matrix(X_dense_test), MFP_test]).tocsr()
oof, tp, fm, bi = cv_boost(
    lambda f, tr, va: (Xc_tr[tr], Xc_tr[va], Xc_te), y, fold_id, "lgb", None)
print("  LightGBM(+Morgan) best_iters:", bi)
register_result("baseline_lgbm_desc_morgan", "descriptors+custom+morgan", "LightGBM",
                Xc_tr.shape[1], fm, 0.0, {}, "-", oof=oof, test_pred=tp)

print("\n参考（既存NB qm9_gap_prediction.ipynb の自己出力）: 記述子+LGBM≈0.2195 / 記述子+Morgan+LGBM≈0.2098")
display(pd.DataFrame(RESULTS)[["strategy", "model", "n_features", "cv_mae_mean", "cv_mae_std"]])

[登録] baseline_median: CV MAE = 1.0753 ± 0.0139
[登録] baseline_ridge: CV MAE = 0.3384 ± 0.0048
[登録] baseline_rf: CV MAE = 0.2202 ± 0.0026
  LightGBM best_iters: [500, 500, 499, 500, 500]
[登録] baseline_lgbm_desc: CV MAE = 0.2255 ± 0.0036
  LightGBM(+Morgan) best_iters: [500, 500, 500, 497, 499]
[登録] baseline_lgbm_desc_morgan: CV MAE = 0.2227 ± 0.0041

参考（既存NB qm9_gap_prediction.ipynb の自己出力）: 記述子+LGBM≈0.2195 / 記述子+Morgan+LGBM≈0.2098


,strategy,model,n_features,cv_mae_mean,cv_mae_std
0,baseline_median,median,0,1.075254,0.013897
1,baseline_ridge,Ridge,217,0.338373,0.004804
2,baseline_rf,RandomForest,217,0.220171,0.002631
3,baseline_lgbm_desc,LightGBM,217,0.225508,0.003618
4,baseline_lgbm_desc_morgan,LightGBM,2265,0.222708,0.004148


## 11. 方針1：重要RDKit特徴量 ＋ ブースティング

**何をする処理か**：記述子＋独自特徴量から**重要特徴量を選び**、LightGBM/XGBoost を Optuna で調整する。
手順は仕様どおり：(1)定数列削除→(2)完全重複列削除（§9で実施済み）→(3)欠損確認→(4)全特徴量で初期モデル→
(5)重要度算出（Permutation / SHAP / gain）→(6)上位特徴量数を変えて CV MAE 比較→(7)最良数を採用→(8)HP探索。

**なぜ必要か**：全組合せ探索を避けつつ、寄与の大きい特徴に絞って過学習と学習時間を抑えるため。

**リーク上の注意**：重要度は**各学習fold内**（外側検証foldを含まない内部分割）で算出し、fold ごとに独自の上位特徴を選ぶ。
特徴量数 k は共有CVで選ぶため、その CV MAE はやや楽観側になり得る（§14で他方針と相対比較する）。
重要度の優先順位は Permutation ≈ SHAP ＞ gain（gain 単独では選ばない）。

**`best_iter` の読み方**：各foldで early stopping 後の採用木数 `best_iter` を出力する。QM9のgapは滑らかで検証MAEが
木数とともに緩やかに改善し続けるため、`best_iter` が探索上限（`boost_nest_range` の上端）に張り付くことがある。
その場合は「収束した」ではなく「木数上限が律速（さらに木を増やせば僅かに改善余地）」と解釈する。既存NBのように
上限張り付きを収束と混同しないための可視化であり、上限は計算量とのトレードオフで設定している。

In [13]:
name2idx = {n: i for i, n in enumerate(FEAT_NAMES)}

def combined_ranking(gain, perm, shap_imp, names):
    """Permutation≈SHAP＞gain の優先で特徴量を降順ソートし、名前リストとスコアを返す。"""
    def norm(x):
        r = ss.rankdata(np.asarray(x, float)); return r / r.max()
    primary = 0.5 * norm(perm) + 0.5 * norm(shap_imp)
    score = primary + 1e-6 * norm(gain)  # gain は微小なタイブレークのみ
    order = np.argsort(-score)
    return [names[i] for i in order], score

# --- fold ごとの重要度ランキング（初期モデル＝全特徴 LightGBM）---
fold_rank: Dict[int, List[str]] = {}
fold_importance_tables = []
for f, (tr, va) in enumerate(iter_folds(fold_id, N_SPLITS)):
    xi, xv, yi, yv = train_test_split(X_dense[tr], y[tr], test_size=0.15, random_state=SEED + f)
    m = make_lgb()
    m.fit(xi, yi, eval_set=[(xv, yv)], eval_metric="mae",
          callbacks=[lgb.early_stopping(CFG["early_stopping_rounds"], verbose=False), lgb.log_evaluation(0)])
    gain = m.booster_.feature_importance("gain")
    perm = permutation_importance(m, xv, yv, n_repeats=CFG["s1_perm_repeats"], random_state=SEED,
                                  scoring="neg_mean_absolute_error", n_jobs=-1).importances_mean
    sh_sample = xi[:CFG["s1_shap_sample"]]
    shap_imp = np.abs(shap.TreeExplainer(m).shap_values(sh_sample)).mean(axis=0)
    names, score = combined_ranking(gain, perm, shap_imp, FEAT_NAMES)
    fold_rank[f] = names
    fold_importance_tables.append(pd.DataFrame({"feature": FEAT_NAMES, "gain": gain,
                                                "perm": perm, "shap": shap_imp, "score": score,
                                                "fold": f}))
    print(f"  fold{f}: best_iter={m.best_iteration_}  top5={names[:5]}")

# 統合ランキング（全foldの平均順位）
imp_all = pd.concat(fold_importance_tables)
rank_tbl = imp_all.copy()
rank_tbl["rank"] = rank_tbl.groupby("fold")["score"].rank(ascending=False)
agg_rank = rank_tbl.groupby("feature")["rank"].mean().sort_values()
agg_rank.to_csv(RESULTS_DIR / "s1_aggregated_importance.csv")
imp_all.to_csv(RESULTS_DIR / "s1_fold_importance.csv", index=False)
print("統合重要度 上位10:", list(agg_rank.index[:10]))

  fold0: best_iter=500  top5=['n_sp2', 'n_conj_bond', 'BCUT2D_MRHI', 'FractionCSP3', 'fr_aldehyde']
  fold1: best_iter=500  top5=['n_sp2', 'n_conj_bond', 'FractionCSP3', 'BCUT2D_MRHI', 'fr_aldehyde']
  fold2: best_iter=499  top5=['n_sp2', 'n_conj_bond', 'FractionCSP3', 'BCUT2D_MRHI', 'fr_aldehyde']
  fold3: best_iter=500  top5=['n_sp2', 'n_conj_bond', 'BCUT2D_MRHI', 'FractionCSP3', 'fr_aldehyde']
  fold4: best_iter=500  top5=['n_sp2', 'n_conj_bond', 'BCUT2D_MRHI', 'FractionCSP3', 'fr_aldehyde']
統合重要度 上位10: ['n_sp2', 'n_conj_bond', 'BCUT2D_MRHI', 'FractionCSP3', 'fr_aldehyde', 'fr_ketone', 'BertzCT', 'HallKierAlpha', 'r_multiple', 'r_conj_bond']


In [14]:
def topk_idx(f: int, k) -> List[int]:
    names = fold_rank[f] if k == "all" else fold_rank[f][:k]
    return [name2idx[n] for n in names]

def s1_oof(k, params, kind="lgb"):
    """fold内 top-k 選択でブースティングをCVし、oof, test_pred, fold_mae を返す。"""
    oof = np.zeros(len(y)); test_pred = np.zeros(X_dense_test.shape[0]); fm = []
    for f, (tr, va) in enumerate(iter_folds(fold_id, N_SPLITS)):
        cols = topk_idx(f, k)
        Xtr, Xva, Xte = X_dense[tr][:, cols], X_dense[va][:, cols], X_dense_test[:, cols]
        model = make_lgb(params, SEED) if kind == "lgb" else make_xgb(params, SEED)
        fit_boost_es(model, Xtr, y[tr], SEED + f)
        oof[va] = boost_predict(model, Xva)
        test_pred += boost_predict(model, Xte) / N_SPLITS
        fm.append(mean_absolute_error(y[va], oof[va]))
    return oof, test_pred, fm

# --- (6)(7) 特徴量数スイープ ---
k_scores = {}
for k in CFG["s1_feature_counts"]:
    _, _, fm = s1_oof(k, None, "lgb")
    k_scores[str(k)] = float(np.mean(fm))
    print(f"  k={k}: CV MAE = {k_scores[str(k)]:.4f}")
best_k_str = min(k_scores, key=k_scores.get)
best_k = "all" if best_k_str == "all" else int(best_k_str)
json.dump({"k_scores": k_scores, "best_k": best_k_str}, open(RESULTS_DIR / "s1_kscan.json", "w"), indent=2)
print("採用特徴量数 best_k =", best_k)

  k=20: CV MAE = 0.2426
  k=80: CV MAE = 0.2253
  k=all: CV MAE = 0.2256
採用特徴量数 best_k = 80


In [15]:
# --- (8) Optuna ハイパーパラメータ探索（LightGBM / XGBoost）目的=5-fold CV平均MAE ---
def s1_obj_lgb(trial):
    p = dict(
        n_estimators=trial.suggest_int("n_estimators", CFG["boost_nest_range"][0], CFG["boost_nest_range"][1], step=100),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        num_leaves=trial.suggest_int("num_leaves", 15, 255),
        max_depth=trial.suggest_int("max_depth", 3, 12),
        min_child_samples=trial.suggest_int("min_child_samples", 5, 80),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        subsample_freq=1,
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
    )
    _, _, fm = s1_oof(best_k, p, "lgb")
    return float(np.mean(fm))

def s1_obj_xgb(trial):
    p = dict(
        n_estimators=trial.suggest_int("n_estimators", CFG["boost_nest_range"][0], CFG["boost_nest_range"][1], step=100),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        max_depth=trial.suggest_int("max_depth", 3, 12),
        min_child_weight=trial.suggest_float("min_child_weight", 1.0, 20.0),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
    )
    _, _, fm = s1_oof(best_k, p, "xgb")
    return float(np.mean(fm))

t0 = time.time()
study1_lgb = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
study1_lgb.optimize(s1_obj_lgb, n_trials=CFG["s1_optuna_trials"], show_progress_bar=True)
study1_xgb = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
study1_xgb.optimize(s1_obj_xgb, n_trials=CFG["s1_optuna_trials"], show_progress_bar=True)

best_kind, best_study = ("lgb", study1_lgb) if study1_lgb.best_value <= study1_xgb.best_value else ("xgb", study1_xgb)
print(f"方針1 最良モデル: {best_kind}  (LGB={study1_lgb.best_value:.4f} / XGB={study1_xgb.best_value:.4f})")

oof1, test1, fm1 = s1_oof(best_k, best_study.best_params, best_kind)
assert np.isfinite(oof1).all(), "OOFに欠損/無限"
n_feat1 = len(FEAT_NAMES) if best_k == "all" else best_k
sub1 = make_submission(test1, "submission_strategy1.csv", test_df)
register_result("strategy1", f"desc+custom top{best_k}", best_kind.upper(), n_feat1, fm1, time.time() - t0,
                best_study.best_params, "submission_strategy1.csv", oof=oof1, test_pred=test1)

Best trial: 1. Best value: 0.217663: 100%|██████████| 6/6 [01:41<00:00, 16.95s/it]


方針1 最良モデル: xgb  (LGB=0.2229 / XGB=0.2177)
保存: submission_strategy1.csv  (4000行)
[登録] strategy1: CV MAE = 0.2177 ± 0.0043


{'strategy': 'strategy1',
 'feature_set': 'desc+custom top80',
 'model': 'XGB',
 'n_features': 80,
 'cv_mae_mean': 0.21766328752056707,
 'cv_mae_std': 0.004271167536540744,
 'fold1_mae': 0.22302580666241598,
 'fold2_mae': 0.21902676061646464,
 'fold3_mae': 0.21248361217004155,
 'fold4_mae': 0.22092585116236899,
 'fold5_mae': 0.21285440699154415,
 'training_time': 240.55673122406006,
 'best_params': '{"n_estimators": 500, "learning_rate": 0.041918146970289824, "max_depth": 8, "min_child_weight": 11.324334333308267, "subsample": 0.8804477878014645, "colsample_bytree": 0.8561872870410456, "reg_alpha": 0.30111222251271, "reg_lambda": 0.05062523846238371}',
 'submission_path': 'submission_strategy1.csv'}

## 12. 方針2：PCA ＋ RBF-SVR

**何をする処理か**：方針1と同じ「記述子＋独自特徴量」（Morgan は使わない）に対し、
`SimpleImputer→StandardScaler→PCA→SVR(rbf)` の **Pipeline** を各fold内でfitしてCVする。
PCA は累積寄与率（0.90/0.95/0.97/0.99）と主成分数（20/40/80/120）を候補にし、SVR の `C`/`gamma`/`epsilon` を Optuna（対数）で探索する。

**なぜ必要か**：非線形カーネルSVRは中規模・連続特徴で強い。Scaler/PCA を **Pipeline** に入れることで、
標準化・次元圧縮を**各fold学習データだけ**でfitし、リークを防ぐ。

**リーク上の注意**：Scaler/PCA を全データにfitしてからCVしない。必ず Pipeline 内でfoldごとにfitする。
**計算時間**：RBF-SVR は O(n²) 級。QUICK では各foldの学習を `s2_svr_max_train` 件にサブサンプルして時間を抑える（fold検証・テスト推論は全件）。

In [16]:
# 有効な PCA 設定（特徴数・fold学習サンプル数を超えるものは除外）
min_fold_train = len(y) - int(np.bincount(fold_id).max())  # 最小の学習fold件数
max_nc = min(len(FEAT_NAMES), min_fold_train)
pca_settings = [("var", v) for v in CFG["s2_pca_var"]] + \
               [("nc", k) for k in CFG["s2_pca_ncomp"] if k <= max_nc]
print("PCA候補:", pca_settings, "| max_nc:", max_nc)

def build_pipe(pca_kind, pca_val, C, gamma, epsilon):
    n_comp = pca_val if pca_kind == "var" else int(pca_val)
    return Pipeline([("imputer", SimpleImputer(strategy="median")),
                     ("scaler", StandardScaler()),
                     ("pca", PCA(n_components=n_comp, random_state=SEED)),
                     ("svr", SVR(kernel="rbf", C=C, gamma=gamma, epsilon=epsilon))])

s2_timing = []
def s2_cv(pca_kind, pca_val, C, gamma, epsilon, subsample=None):
    """Pipeline を各fold内でfitしてCV。oof,test_pred,fold_mae,fold時間 を返す。"""
    oof = np.zeros(len(y)); test_pred = np.zeros(X_dense_test.shape[0]); fm = []; ft = []
    for f, (tr, va) in enumerate(iter_folds(fold_id, N_SPLITS)):
        tr_use = tr
        if subsample is not None and len(tr) > subsample:
            rng = np.random.RandomState(SEED + f)
            tr_use = rng.choice(tr, size=subsample, replace=False)
        t0 = time.time()
        pipe = build_pipe(pca_kind, pca_val, C, gamma, epsilon)
        pipe.fit(X_dense[tr_use], y[tr_use])
        ft.append(time.time() - t0)
        oof[va] = pipe.predict(X_dense[va])
        test_pred += pipe.predict(X_dense_test) / N_SPLITS
        fm.append(mean_absolute_error(y[va], oof[va]))
    return oof, test_pred, fm, ft

def s2_obj(trial):
    pk, pv = trial.suggest_categorical("pca", [f"{k}:{v}" for k, v in pca_settings]).split(":")
    pv = float(pv) if pk == "var" else int(pv)
    C = trial.suggest_float("C", 1e-1, 1e4, log=True)
    gamma_mode = trial.suggest_categorical("gamma_mode", ["scale", "value"])
    gamma = "scale" if gamma_mode == "scale" else trial.suggest_float("gamma", 1e-5, 1e0, log=True)
    epsilon = trial.suggest_float("epsilon", 1e-3, 10 ** -0.3, log=True)
    t0 = time.time()
    _, _, fm, ft = s2_cv(pk, pv, C, gamma, epsilon, subsample=CFG["s2_svr_max_train"])
    s2_timing.append(dict(trial=trial.number, pca=f"{pk}:{pv}", C=C, gamma=str(gamma),
                          epsilon=epsilon, mae=float(np.mean(fm)),
                          fold_time=[round(t, 2) for t in ft], total_time=round(time.time() - t0, 2)))
    return float(np.mean(fm))

t0 = time.time()
study2 = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
study2.optimize(s2_obj, n_trials=CFG["s2_optuna_trials"], show_progress_bar=True)
pd.DataFrame(s2_timing).to_csv(RESULTS_DIR / "s2_timing.csv", index=False)

# PCA 設定別の最良MAE
timing_df = pd.DataFrame(s2_timing)
print("PCA設定別 最良CV MAE:")
display(timing_df.groupby("pca")["mae"].min().sort_values())
print("方針2 best params:", study2.best_params, " best MAE:", round(study2.best_value, 4))

PCA候補: [('var', 0.95), ('var', 0.99), ('nc', 40), ('nc', 120)] | max_nc: 217


Best trial: 9. Best value: 0.284747: 100%|██████████| 10/10 [04:51<00:00, 29.14s/it]

PCA設定別 最良CV MAE:


pca
nc:120      0.284747
nc:40       0.317353
var:0.99    0.317367
var:0.95    0.327604
Name: mae, dtype: float64

方針2 best params: {'pca': 'nc:120', 'C': 1.0400371697287312, 'gamma_mode': 'value', 'gamma': 0.0029378017798391844, 'epsilon': 0.04372457057822113}  best MAE: 0.2847


In [17]:
# 最良設定で全件学習して確定（QUICKのサブサンプルは探索時のみ。最終は全件でfit）
bp = study2.best_params
pk, pv = bp["pca"].split(":"); pv = float(pv) if pk == "var" else int(pv)
gamma = "scale" if bp["gamma_mode"] == "scale" else bp["gamma"]
oof2, test2, fm2, ft2 = s2_cv(pk, pv, bp["C"], gamma, bp["epsilon"], subsample=None)
# 採用主成分数・累積寄与率（fold0で確認）
p0 = build_pipe(pk, pv, bp["C"], gamma, bp["epsilon"])
tr0 = next(iter_folds(fold_id, N_SPLITS))[0]
p0.fit(X_dense[tr0], y[tr0])
n_pca = p0.named_steps["pca"].n_components_
cum_var = float(p0.named_steps["pca"].explained_variance_ratio_.sum())
print(f"採用主成分数={n_pca}  累積寄与率={cum_var:.3f}  最終CV MAE={np.mean(fm2):.4f}")
assert np.isfinite(oof2).all()
sub2 = make_submission(test2, "submission_strategy2.csv", test_df)
register_result("strategy2", "desc+custom+PCA", "RBF-SVR", n_pca, fm2, time.time() - t0,
                {**bp, "n_pca": n_pca, "cum_var": cum_var}, "submission_strategy2.csv", oof=oof2, test_pred=test2)

採用主成分数=120  累積寄与率=0.997  最終CV MAE=0.2392
保存: submission_strategy2.csv  (4000行)
[登録] strategy2: CV MAE = 0.2392 ± 0.0043


{'strategy': 'strategy2',
 'feature_set': 'desc+custom+PCA',
 'model': 'RBF-SVR',
 'n_features': 120,
 'cv_mae_mean': 0.23920046747294169,
 'cv_mae_std': 0.004328494186195426,
 'fold1_mae': 0.2428906985021762,
 'fold2_mae': 0.23428436307708989,
 'fold3_mae': 0.23666579691973724,
 'fold4_mae': 0.24568497780743,
 'fold5_mae': 0.23647650105827506,
 'training_time': 383.608784198761,
 'best_params': '{"pca": "nc:120", "C": 1.0400371697287312, "gamma_mode": "value", "gamma": 0.0029378017798391844, "epsilon": 0.04372457057822113, "n_pca": 120, "cum_var": 0.9970780611038208}',
 'submission_path': 'submission_strategy2.csv'}

## 13. 方針3：大域RDKit ＋ 独自特徴量 ＋ Morgan ＋ ブースティング

**何をする処理か**：標準記述子から `fr_*` と `FpDensityMorgan1/2/3` を除外（Morganとの情報重複を抑制）し、
定数・重複・欠損過多列を除去。高相関列は「残す/しきい値0.95・0.98・0.995で削除」を CV MAE で比較。独自特徴量を加え、
Morgan（count/binary、radius2/3、低頻度bit削除条件）も CV MAE で比較。最後に A=記述子+独自 / B=Morganのみ /
C=全部 を同一foldで比較し、C の改善を確認してから Optuna で LightGBM/XGBoost を調整する。

**なぜ必要か**：解釈しやすい大域特徴と、部分構造を表す高次元FPを組み合わせて表現力を上げるため。

**リーク上の注意**：相関・低頻度bitの判定は**教師なし**（目的変数不使用）なので全trainで実施可。疎行列を維持し dense 化を避ける。
高次元FPの過学習は列サンプリング（`colsample_bytree`）と正則化（`reg_alpha/lambda`）で抑える。

In [18]:
# 大域RDKit記述子: fr_* と FpDensityMorgan1/2/3 を除外
drop_desc = [c for c in desc_train.columns
             if c.startswith("fr_") or c in {"FpDensityMorgan1", "FpDensityMorgan2", "FpDensityMorgan3"}]
g_desc_tr = desc_train.drop(columns=drop_desc)
g_desc_te = desc_test.drop(columns=drop_desc)
print(f"標準記述子から除外: {len(drop_desc)} 列（fr_*={sum(c.startswith('fr_') for c in drop_desc)}, FpDensityMorgan=3）")

# 記述子＋独自 → 教師なしクリーニング
g_tr = pd.concat([g_desc_tr, cust_train], axis=1)
g_te = pd.concat([g_desc_te, cust_test], axis=1)
g_tr, g_te, gc, gd = clean_dense(g_tr, g_te)
print(f"方針3 記述子+独自: 採用 {g_tr.shape[1]} 列（定数{len(gc)}/重複{len(gd)}削除）")

def sweep_cv(Xtr, Xte, y, n_splits_used):
    """設定比較用の軽量CV（先頭 n_splits_used fold・固定LightGBM）。平均MAEを返す。"""
    fm = []
    params = dict(n_estimators=(400 if QUICK else 1500))
    for f, (tr, va) in enumerate(iter_folds(fold_id, N_SPLITS)):
        if f >= n_splits_used:
            break
        model = make_lgb(params)
        fit_boost_es(model, Xtr[tr], y[tr], SEED + f)
        pred = boost_predict(model, Xtr[va])
        fm.append(mean_absolute_error(y[va], pred))
    return float(np.mean(fm))

# --- 高相関列の扱いを比較（相関はtrainのみ・教師なし）---
Xg = g_tr.to_numpy(np.float32)
corr = np.nan_to_num(np.corrcoef(Xg, rowvar=False))
def drop_highcorr(thr):
    if thr is None:
        return list(range(g_tr.shape[1]))
    keep = []
    dropped = set()
    for i in range(g_tr.shape[1]):
        if i in dropped:
            continue
        keep.append(i)
        high = np.where(np.abs(corr[i]) > thr)[0]
        for j in high:
            if j > i:
                dropped.add(j)
    return keep

corr_scores = {}
for thr in CFG["s3_corr_thresholds"]:
    cols = drop_highcorr(thr)
    corr_scores[str(thr)] = sweep_cv(Xg[:, cols], None, y, CFG["sweep_n_splits"])
    print(f"  高相関しきい値 {thr}: {len(cols)}列  CV MAE={corr_scores[str(thr)]:.4f}")
best_thr_key = min(corr_scores, key=corr_scores.get)
best_thr = None if best_thr_key == "None" else float(best_thr_key)
g_cols = drop_highcorr(best_thr)
Xg_tr = Xg[:, g_cols]; Xg_te = g_te.to_numpy(np.float32)[:, g_cols]
json.dump(corr_scores, open(RESULTS_DIR / "s3_corr_scores.json", "w"), indent=2)
print("採用しきい値:", best_thr, "→", len(g_cols), "列")

標準記述子から除外: 88 列（fr_*=85, FpDensityMorgan=3）
方針3 記述子+独自: 採用 158 列（定数8/重複16削除）
  高相関しきい値 0.98: 137列  CV MAE=0.2352
  高相関しきい値 None: 158列  CV MAE=0.2360
採用しきい値: 0.98 → 137 列


In [19]:
# --- Morgan 設定 × 低頻度bit削除条件の比較 ---
def filter_lowfreq(M_tr, M_te, min_df):
    """train上で出現分子数>=min_df のbitのみ残す（教師なし）。"""
    df_count = np.asarray((M_tr > 0).sum(axis=0)).ravel()
    keep = np.where(df_count >= min_df)[0]
    return M_tr[:, keep].tocsr(), M_te[:, keep].tocsr(), keep

morgan_scores = []
best_cfg = None
for (kind, radius, nbits) in CFG["s3_morgan_variants"]:
    count = (kind == "count")
    Mtr = cached_morgan(f"train_{kind[0]}{radius}_{nbits}", train_mols, radius, nbits, count)
    Mte = cached_morgan(f"test_{kind[0]}{radius}_{nbits}", test_mols, radius, nbits, count)
    for min_df in CFG["s3_lowfreq_min_df"]:
        Mtr_f, Mte_f, keep = filter_lowfreq(Mtr, Mte, min_df)
        Xc = sp.hstack([sp.csr_matrix(Xg_tr), Mtr_f]).tocsr()
        mae = sweep_cv(Xc, None, y, CFG["sweep_n_splits"])
        morgan_scores.append(dict(kind=kind, radius=radius, nbits=nbits, min_df=min_df,
                                  n_bits_kept=int(len(keep)), n_features=int(Xc.shape[1]), mae=mae))
        print(f"  {kind} r{radius} {nbits} min_df={min_df}: bits={len(keep)}  CV MAE={mae:.4f}")
        if best_cfg is None or mae < best_cfg["mae"]:
            best_cfg = morgan_scores[-1]
pd.DataFrame(morgan_scores).to_csv(RESULTS_DIR / "s3_morgan_scores.csv", index=False)
print("採用 Morgan 設定:", best_cfg)

[cache hit] morgan_train_c2_2048
[cache hit] morgan_test_c2_2048
  count r2 2048 min_df=1: bits=2048  CV MAE=0.2321
  count r2 2048 min_df=5: bits=2048  CV MAE=0.2321
採用 Morgan 設定: {'kind': 'count', 'radius': 2, 'nbits': 2048, 'min_df': 1, 'n_bits_kept': 2048, 'n_features': 2185, 'mae': 0.23212686032217217}


In [20]:
# 採用設定で最終 Morgan を用意
kind, radius, nbits, min_df = best_cfg["kind"], best_cfg["radius"], best_cfg["nbits"], best_cfg["min_df"]
count = (kind == "count")
Mtr = cached_morgan(f"train_{kind[0]}{radius}_{nbits}", train_mols, radius, nbits, count)
Mte = cached_morgan(f"test_{kind[0]}{radius}_{nbits}", test_mols, radius, nbits, count)
Mtr_f, Mte_f, keep_bits = filter_lowfreq(Mtr, Mte, min_df)

XA_tr = sp.csr_matrix(Xg_tr);                    XA_te = sp.csr_matrix(Xg_te)          # A: 記述子+独自
XB_tr = Mtr_f;                                   XB_te = Mte_f                          # B: Morganのみ
XC_tr = sp.hstack([XA_tr, Mtr_f]).tocsr();       XC_te = sp.hstack([XA_te, Mte_f]).tocsr()  # C: 全部

# --- A / B / C 比較（同一fold・固定モデル）---
abc = {}
for tag, Xt in [("A: 記述子+独自", XA_tr), ("B: Morganのみ", XB_tr), ("C: 全部", XC_tr)]:
    abc[tag] = sweep_cv(Xt, None, y, N_SPLITS)
    print(f"  {tag}: CV MAE={abc[tag]:.4f}  (features={Xt.shape[1]})")
json.dump(abc, open(RESULTS_DIR / "s3_abc.json", "w"), ensure_ascii=False, indent=2)
print("C が A・B を改善:", abc["C: 全部"] < min(abc["A: 記述子+独自"], abc["B: Morganのみ"]))

[cache hit] morgan_train_c2_2048
[cache hit] morgan_test_c2_2048
  A: 記述子+独自: CV MAE=0.2352  (features=137)
  B: Morganのみ: CV MAE=0.3267  (features=2048)
  C: 全部: CV MAE=0.2312  (features=2185)
C が A・B を改善: True


In [21]:
# --- Optuna（LightGBM/XGBoost）: 高次元FPの過学習を列サンプリング＋正則化で抑制 ---
def s3_cv(params, kind="lgb"):
    oof = np.zeros(len(y)); test_pred = np.zeros(XC_te.shape[0]); fm = []
    for f, (tr, va) in enumerate(iter_folds(fold_id, N_SPLITS)):
        model = make_lgb(params, SEED) if kind == "lgb" else make_xgb(params, SEED)
        fit_boost_es(model, XC_tr[tr], y[tr], SEED + f)
        oof[va] = boost_predict(model, XC_tr[va])
        test_pred += boost_predict(model, XC_te) / N_SPLITS
        fm.append(mean_absolute_error(y[va], oof[va]))
    return oof, test_pred, fm

def s3_obj_lgb(trial):
    p = dict(
        n_estimators=trial.suggest_int("n_estimators", CFG["boost_nest_range"][0], CFG["boost_nest_range"][1], step=100),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        num_leaves=trial.suggest_int("num_leaves", 15, 255),
        max_depth=trial.suggest_int("max_depth", 3, 12),
        min_child_samples=trial.suggest_int("min_child_samples", 5, 100),
        subsample=trial.suggest_float("subsample", 0.5, 1.0), subsample_freq=1,
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.2, 0.9),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 20.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 20.0, log=True),
    )
    return float(np.mean(s3_cv(p, "lgb")[2]))

def s3_obj_xgb(trial):
    p = dict(
        n_estimators=trial.suggest_int("n_estimators", CFG["boost_nest_range"][0], CFG["boost_nest_range"][1], step=100),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        max_depth=trial.suggest_int("max_depth", 3, 12),
        min_child_weight=trial.suggest_float("min_child_weight", 1.0, 30.0),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.2, 0.9),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 20.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 20.0, log=True),
    )
    return float(np.mean(s3_cv(p, "xgb")[2]))

t0 = time.time()
study3_lgb = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
study3_lgb.optimize(s3_obj_lgb, n_trials=CFG["s3_optuna_trials"], show_progress_bar=True)
study3_xgb = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
study3_xgb.optimize(s3_obj_xgb, n_trials=CFG["s3_optuna_trials"], show_progress_bar=True)
best3_kind, best3 = ("lgb", study3_lgb) if study3_lgb.best_value <= study3_xgb.best_value else ("xgb", study3_xgb)
print(f"方針3 最良モデル: {best3_kind}  (LGB={study3_lgb.best_value:.4f} / XGB={study3_xgb.best_value:.4f})")

oof3, test3, fm3 = s3_cv(best3.best_params, best3_kind)
assert np.isfinite(oof3).all()
sub3 = make_submission(test3, "submission_strategy3.csv", test_df)
register_result("strategy3", f"desc+custom+morgan({kind}r{radius},df>={min_df})", best3_kind.upper(),
                XC_tr.shape[1], fm3, time.time() - t0,
                {**best3.best_params, "morgan": best_cfg, "corr_thr": best_thr},
                "submission_strategy3.csv", oof=oof3, test_pred=test3)

Best trial: 1. Best value: 0.225223: 100%|██████████| 6/6 [07:07<00:00, 71.28s/it] 


方針3 最良モデル: xgb  (LGB=0.2295 / XGB=0.2252)
保存: submission_strategy3.csv  (4000行)
[登録] strategy3: CV MAE = 0.2252 ± 0.0050


{'strategy': 'strategy3',
 'feature_set': 'desc+custom+morgan(countr2,df>=1)',
 'model': 'XGB',
 'n_features': 2185,
 'cv_mae_mean': 0.22522269748245302,
 'cv_mae_std': 0.0050257426878084835,
 'fold1_mae': 0.23065551809923093,
 'fold2_mae': 0.223637086970341,
 'fold3_mae': 0.22242373972295676,
 'fold4_mae': 0.23125132516763958,
 'fold5_mae': 0.21814581745209694,
 'training_time': 699.5031771659851,
 'best_params': '{"n_estimators": 500, "learning_rate": 0.041918146970289824, "max_depth": 8, "min_child_weight": 16.75819450873367, "subsample": 0.8804477878014645, "colsample_bytree": 0.6986622018574639, "reg_alpha": 0.4626699390692671, "reg_lambda": 0.06801950791585142, "morgan": {"kind": "count", "radius": 2, "nbits": 2048, "min_df": 1, "n_bits_kept": 2048, "n_features": 2185, "mae": 0.23212686032217217}, "corr_thr": 0.98}',
 'submission_path': 'submission_strategy3.csv'}

## 14. 3方針比較

**何をする処理か**：ベースラインと3方針を、strategy / feature_set / model / n_features / cv_mae_mean / cv_mae_std /
fold別MAE / training_time / best_params / submission_path の形式で一覧化し CSV 保存する。

**実行結果の読み方**：`cv_mae_mean` が小さいほど良い。`cv_mae_std` はfold間のばらつき。方針間の差が std に対して小さければ僅差。

In [22]:
cmp = pd.DataFrame(RESULTS)
cols = ["strategy", "feature_set", "model", "n_features", "cv_mae_mean", "cv_mae_std",
        "fold1_mae", "fold2_mae", "fold3_mae", "fold4_mae", "fold5_mae",
        "training_time", "best_params", "submission_path"]
cmp = cmp[[c for c in cols if c in cmp.columns]].sort_values("cv_mae_mean").reset_index(drop=True)
cmp.to_csv(RESULTS_DIR / "strategy_comparison.csv", index=False)
display(cmp.drop(columns=["best_params"]))
best_row = cmp.iloc[0]
print(f"\n最良: {best_row['strategy']}  CV MAE = {best_row['cv_mae_mean']:.4f} ± {best_row['cv_mae_std']:.4f}")

,strategy,feature_set,model,n_features,cv_mae_mean,cv_mae_std,fold1_mae,fold2_mae,fold3_mae,fold4_mae,fold5_mae,training_time,submission_path
0,strategy1,desc+custom top80,XGB,80,0.217663,0.004271,0.223026,0.219027,0.212484,0.220926,0.212854,240.556731,submission_strategy1.csv
1,baseline_rf,descriptors+custom,RandomForest,217,0.220171,0.002631,0.224772,0.219522,0.218239,0.221067,0.217256,123.485882,-
2,baseline_lgbm_desc_morgan,descriptors+custom+morgan,LightGBM,2265,0.222708,0.004148,0.227408,0.220360,0.219678,0.227998,0.218096,0.000000,-
3,strategy3,"desc+custom+morgan(countr2,df>=1)",XGB,2185,0.225223,0.005026,0.230656,0.223637,0.222424,0.231251,0.218146,699.503177,submission_strategy3.csv
4,baseline_lgbm_desc,descriptors+custom,LightGBM,217,0.225508,0.003618,0.229246,0.222921,0.222313,0.230559,0.222500,0.000000,-
5,strategy2,desc+custom+PCA,RBF-SVR,120,0.239200,0.004328,0.242891,0.234284,0.236666,0.245685,0.236477,383.608784,submission_strategy2.csv
6,baseline_ridge,descriptors+custom,Ridge,217,0.338373,0.004804,0.341449,0.338834,0.342318,0.340208,0.329058,0.927711,-
7,baseline_median,-,median,0,1.075254,0.013897,1.096162,1.080894,1.075748,1.069823,1.053644,0.025824,-



最良: strategy1  CV MAE = 0.2177 ± 0.0043


## 15. ブレンド参考実験（正式方針ではない）

**何をする処理か**：方針1・2・3 の OOF 予測を、`重み>=0` かつ `合計=1` の制約で組み合わせ、MAE を確認する。
組合せは 1+2 / 1+3 / 2+3 / 1+2+3。同じOOFで重みを最適化して同じOOFで測ると楽観的になるため、
**(a) 全OOF最適化（楽観・参考値）** と **(b) fold外し推定（fold毎に他foldで重みを推定して適用した控えめな推定）** の両方を出す。

**なぜ必要か**：既存NBのブレンドは片方に偏り情報が薄かった。ここでは参考実験として明示し、
**方針3を置き換えない**。提出は §16 の3方針を用いる。

In [23]:
oofs = {"s1": oof1, "s2": oof2, "s3": oof3}
tests = {"s1": test1, "s2": test2, "s3": test3}

def opt_weights(oof_list, y):
    """重み>=0, 合計=1 の制約下で OOF-MAE を最小化する重みを返す。"""
    k = len(oof_list)
    P = np.vstack(oof_list).T
    def loss(w):
        return mean_absolute_error(y, P @ w)
    cons = ({"type": "eq", "fun": lambda w: w.sum() - 1.0},)
    res = minimize(loss, np.full(k, 1.0 / k), method="SLSQP",
                   bounds=[(0.0, 1.0)] * k, constraints=cons)
    return res.x

combos = [("1+2", ["s1", "s2"]), ("1+3", ["s1", "s3"]), ("2+3", ["s2", "s3"]),
          ("1+2+3", ["s1", "s2", "s3"])]
rows = []
for name, keys in combos:
    ol = [oofs[k] for k in keys]
    # (a) 全OOF最適化（楽観・参考）
    w_opt = opt_weights(ol, y)
    mae_opt = mean_absolute_error(y, np.vstack(ol).T @ w_opt)
    # (b) fold外し推定（各foldは他foldで重み推定）
    oof_nested = np.zeros(len(y))
    for f, (tr, va) in enumerate(iter_folds(fold_id, N_SPLITS)):
        w = opt_weights([o[tr] for o in ol], y[tr])
        oof_nested[va] = np.vstack([o[va] for o in ol]).T @ w
    mae_nested = mean_absolute_error(y, oof_nested)
    rows.append(dict(combo=name, weights=np.round(w_opt, 3).tolist(),
                     mae_oof_optimistic=round(mae_opt, 4), mae_nested=round(mae_nested, 4)))
blend_df = pd.DataFrame(rows)
blend_df.to_csv(RESULTS_DIR / "blend_reference.csv", index=False)
print("※ mae_oof_optimistic は楽観側、mae_nested がより信頼できる参考値")
display(blend_df)

※ mae_oof_optimistic は楽観側、mae_nested がより信頼できる参考値


,combo,weights,mae_oof_optimistic,mae_nested
0,1+2,"[0.777, 0.223]",0.2153,0.2153
1,1+3,"[0.78, 0.22]",0.2169,0.2170
2,2+3,"[0.327, 0.673]",0.2200,0.2200
3,1+2+3,"[0.638, 0.21, 0.152]",0.2150,0.2151


## 16. 提出CSV生成と検証

**何をする処理か**：3方針の提出CSV（§11/12/13で作成済み）を再読込し、行数・列順・SMILES順・欠損/無限/重複を最終検証する。

**実行結果の読み方**：3ファイルすべてで `OK` が出れば提出フォーマット要件（`smiles,gap_eV`・4000行）を満たす。

In [24]:
for path in ["submission_strategy1.csv", "submission_strategy2.csv", "submission_strategy3.csv"]:
    s = pd.read_csv(path)
    assert list(s.columns) == ["smiles", "gap_eV"], f"{path}: 列順不正"
    assert len(s) == len(test_df), f"{path}: 行数不一致"
    assert s["smiles"].tolist() == test_df["smiles"].tolist(), f"{path}: SMILES順不一致"
    assert s["gap_eV"].notna().all() and np.isfinite(s["gap_eV"]).all(), f"{path}: 欠損/無限"
    assert not s.duplicated().any(), f"{path}: 重複行"
    print(f"OK  {path}  ({len(s)}行)  mean={s['gap_eV'].mean():.3f}")

OK  submission_strategy1.csv  (4000行)  mean=6.884
OK  submission_strategy2.csv  (4000行)  mean=6.889
OK  submission_strategy3.csv  (4000行)  mean=6.884


## 17. 最終結果まとめ

**何をする処理か**：比較表から各方針のCV MAEと最良方針を要約表示する。

**用語メモ**：QM9＝最大9重原子(C,N,O,F)の有機分子の量子化学データ／RDKit＝ケモインフォマティクスライブラリ／
SMILES＝分子構造の文字列表記／分子記述子＝分子を数値化した量／フィンガープリント＝部分構造をビット列で表す特徴。

In [25]:
print("=== 各方針 CV MAE（小さいほど良い）===")
for _, r in pd.DataFrame(RESULTS).sort_values("cv_mae_mean").iterrows():
    print(f"  {r['strategy']:28s} {r['model']:10s} n={str(r['n_features']):>5} "
          f"MAE={r['cv_mae_mean']:.4f} ± {r['cv_mae_std']:.4f}")
best = pd.DataFrame(RESULTS).sort_values("cv_mae_mean").iloc[0]
print(f"\n最良方針: {best['strategy']}  ({best['model']})  CV MAE={best['cv_mae_mean']:.4f}")
print("参考: 既存NBベースライン 記述子+LGBM≈0.2195 / 記述子+Morgan+LGBM≈0.2098")

=== 各方針 CV MAE（小さいほど良い）===
  strategy1                    XGB        n=   80 MAE=0.2177 ± 0.0043
  baseline_rf                  RandomForest n=  217 MAE=0.2202 ± 0.0026
  baseline_lgbm_desc_morgan    LightGBM   n= 2265 MAE=0.2227 ± 0.0041
  strategy3                    XGB        n= 2185 MAE=0.2252 ± 0.0050
  baseline_lgbm_desc           LightGBM   n=  217 MAE=0.2255 ± 0.0036
  strategy2                    RBF-SVR    n=  120 MAE=0.2392 ± 0.0043
  baseline_ridge               Ridge      n=  217 MAE=0.3384 ± 0.0048
  baseline_median              median     n=    0 MAE=1.0753 ± 0.0139

最良方針: strategy1  (XGB)  CV MAE=0.2177
参考: 既存NBベースライン 記述子+LGBM≈0.2195 / 記述子+Morgan+LGBM≈0.2098


## 18. 再現方法

**環境構築**
```bash
cd lesson_9/self-code
uv sync           # pyproject.toml の依存（xgboost/optuna/shap/pyarrow/matplotlib 追加済み）を復元
```

**実行**
- 動作確認（軽量）：`QUICK = True`（§2の設定セル）のまま、上から順に全セルを実行。
- フル探索：`QUICK = False` に変更してから全セル実行。または CLI で:
```bash
uv run jupyter nbconvert --to notebook --execute --inplace \
  --ExecutePreprocessor.timeout=-1 qm9_gap_prediction_revised.ipynb
```
（CPUのみ・GPU不要。Apple Silicon 実測の目安：特徴量生成 約50秒/1回（以後キャッシュ）、
LightGBM 1fold学習が木数に比例して 木500で約20秒・木1500で約60秒、RBF-SVR 1fold学習が全件で約35秒・
4000件サブサンプルで約1.4秒。Optuna 1試行＝5fold。**QUICK ≈ 1〜2時間、FULL ≈ 数時間〜一晩**が目安。
さらに短縮するには §2 の `s*_optuna_trials` や `boost_nest_range` を小さくする。）

**best_iter の確認**：各方針の学習ログに出る `best_iter` が探索上限に張り付く場合は「木数律速」。
提出値には影響しないが、精度をさらに詰めるなら `boost_nest_range` の上端を上げる（計算時間は増える）。

**再現性**：seed=8 を Python/NumPy/scikit-learn/LightGBM/XGBoost/Optuna(TPESampler) に統一。
fold は `fold_id`（§5）で全方針共有。特徴量は `cache/` に、OOF/テスト予測・結果JSONは `results/` に保存される
（設定変更時はキャッシュのmetaが不一致になり自動再計算）。

**ライブラリバージョン**（この環境）:

In [26]:
print(json.dumps(LIB_VERSIONS, ensure_ascii=False, indent=2))
json.dump(LIB_VERSIONS, open(RESULTS_DIR / "lib_versions.json", "w"), ensure_ascii=False, indent=2)

{
  "python": "3.11.15",
  "platform": "macOS-14.5-arm64-arm-64bit",
  "machine": "arm64",
  "numpy": "2.4.6",
  "pandas": "3.0.3",
  "scipy": "1.17.1",
  "scikit-learn": "1.9.0",
  "lightgbm": "4.7.0",
  "xgboost": "3.2.0",
  "optuna": "4.9.0",
  "shap": "0.51.0",
  "rdkit": "2026.03.4"
}
